# Prism Eval

Runs the Prism Recipe against baseline on nanoGPT Shakespeare across multiple
seeds and produces the **Prism Score** — how many fewer steps Prism needs to
reach baseline quality.

Score 1.0 = no benefit. Higher = better. There is no expected value: whatever
this prints is the result.

**Read the score against `baseline_best`.** The Prism Score is a ratio, so a
weak baseline inflates it without the method improving. A score marked
*lower bound* means the target was hit at the first eval and the true crossing
is unresolved.

**This notebook writes an artifact to `results/`.** Run cell 2 to download it,
then commit it. Save this notebook **with outputs**. A number without a
committed artifact is not a result.

Runtime: ~45-60 min on an A100 for 3 seeds.

---
*[github.com/timepointai/nanogpt-prism-shakespeare](https://github.com/timepointai/nanogpt-prism-shakespeare) · [Sean McDonald](https://x.com/seanmcdonaldxyz)*

In [ ]:
import os, shutil, subprocess, sys

REPO = '/content/nanogpt-prism'
os.chdir('/content')
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone',
    'https://github.com/timepointai/nanogpt-prism-shakespeare.git',
    REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'tiktoken', 'datasets'], check=True)

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — CPU/MPS, will be slow"}')
print(f'commit: {subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()}\n')

# Streamed so a long run stays auditable while it goes.
p = subprocess.Popen(
    [sys.executable, '-u', 'prism_eval.py',
     '--method=recipe', '--teacher_steps=2000', '--student_steps=5000',
     '--seeds=1337,1338,1339'],
    cwd=f'{REPO}/src', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
rc = p.wait()
if rc != 0:
    raise SystemExit(f'Eval failed (exit {rc}). No artifact written — nothing to publish.')

In [ ]:
# Print the artifact (captured in this notebook's saved outputs) and download it.
import json, glob, os

art = sorted(glob.glob(f'{REPO}/results/recipe_*.json'))[-1]
print(json.dumps(json.load(open(art)), indent=2))
print(f'\nCommit this to results/: {os.path.basename(art)}')

try:
    from google.colab import files
    files.download(art)
except Exception as e:
    print(f'(no Colab download: {e})')